In [1]:
import os
import pandas as pd

BASE = r'C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset'

# === 1. Cargar las 3 fuentes ===
labels    = pd.read_csv(rf'{BASE}\mimic-cxr-2.0.0-chexpert.csv')
metadata  = pd.read_csv(rf'{BASE}\mimic-cxr-2.0.0-metadata.csv')
filenames = pd.read_csv(rf'{BASE}\IMAGE_FILENAMES', header=None, names=['full_path'])

# === 2. Extraer dicom_id desde el path ===
# IMAGE_FILENAMES tiene: files/p10/p10000032/s50414267/02aa804e-bde0afdd-...jpg
# El dicom_id es el nombre del archivo sin .jpg
filenames['dicom_id'] = filenames['full_path'].str.extract(r'/([^/]+)\.jpg$')

# === 3. Merge: metadata + labels + filenames ===
df = (
    metadata[['dicom_id', 'subject_id', 'study_id']]
    .merge(labels, on=['subject_id', 'study_id'], how='left')
    .merge(filenames[['dicom_id', 'full_path']], on='dicom_id', how='left')
)

# === 4. Ordenar columnas como quieres ===
cols = [
    'dicom_id', 'subject_id', 'study_id',
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
    'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
    'Pneumothorax', 'Support Devices', 'full_path'
]
df = df[cols]

# === 5. Guardar ===
out_dir  = rf'{BASE}\dataset-completo-merge'
out_path = rf'{out_dir}\df_completo.csv'
os.makedirs(out_dir, exist_ok=True)
df.to_csv(out_path, index=False)

print(f"Total filas: {len(df):,}")
print(f"Filas con full_path: {df['full_path'].notna().sum():,}")
print(f"Filas con labels:    {df['Atelectasis'].notna().sum():,}")
print(f"Guardado en: {out_path}")
df.head()

Total filas: 377,110
Filas con full_path: 377,110
Filas con labels:    82,830
Guardado en: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo.csv


,dicom_id,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,full_path
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p10/p10000032/s50414267/02aa804e-bde0afd...
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p10/p10000032/s50414267/174413ec-4ec4c1f...
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p10/p10000032/s53189527/2a2277a9-b0ded15...
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p10/p10000032/s53189527/e084de3b-be89b11...
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p10/p10000032/s53911762/68b5c4b1-227d048...


In [2]:
df['full_path'].head()

0    files/p10/p10000032/s50414267/02aa804e-bde0afd...
1    files/p10/p10000032/s50414267/174413ec-4ec4c1f...
2    files/p10/p10000032/s53189527/2a2277a9-b0ded15...
3    files/p10/p10000032/s53189527/e084de3b-be89b11...
4    files/p10/p10000032/s53911762/68b5c4b1-227d048...
Name: full_path, dtype: str

In [1]:
from pathlib import Path
import pandas as pd
 
 
DF_COMPLETO_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_completo.csv"
)
METADATA_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\mimic-cxr-2.0.0-metadata.csv"
)
OUTPUT_PATH = DF_COMPLETO_PATH.with_name("df_completo_view.csv")
 
 
def main() -> None:
    print(f"Leyendo df_completo: {DF_COMPLETO_PATH}")
    df = pd.read_csv(DF_COMPLETO_PATH)
    print(f"  shape: {df.shape}")
 
    print(f"Leyendo metadata:    {METADATA_PATH}")
    meta = pd.read_csv(METADATA_PATH, usecols=[
        "dicom_id", "subject_id", "study_id", "ViewPosition"
    ])
    print(f"  shape: {meta.shape}")
 
    # Idempotencia: si ya existe ViewPosition, la dropeamos antes de re-mergear.
    if "ViewPosition" in df.columns:
        print("  Aviso: ViewPosition ya existía en df_completo, se rehace.")
        df = df.drop(columns=["ViewPosition"])
 
    # Merge estricto por la triple llave; falla si hay duplicados.
    join_keys = ["dicom_id", "subject_id", "study_id"]
    df = df.merge(meta, on=join_keys, how="left", validate="one_to_one")
 
    unmatched = df["ViewPosition"].isna().sum()
    print(f"\nMerge OK. Filas sin match en metadata: {unmatched} / {len(df)}")
 
    # Diagnósticos breves
    print("\nViewPosition (todo df_completo):")
    print(df["ViewPosition"].value_counts(dropna=False).to_string())
 
    bad = df[df["calidad-imagen"] == 0.0]
    print(f"\nViewPosition de las {len(bad)} bad-quality:")
    print(bad["ViewPosition"].value_counts(dropna=False).to_string())
 
    # Escritura a un archivo nuevo, sin tocar df_completo.csv.
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nGuardado: {OUTPUT_PATH}")
    print(f"  shape final: {df.shape}")
 
 
if __name__ == "__main__":
    main()

Leyendo df_completo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo.csv
  shape: (377110, 24)
Leyendo metadata:    C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\mimic-cxr-2.0.0-metadata.csv
  shape: (377110, 4)

Merge OK. Filas sin match en metadata: 15769 / 377110

ViewPosition (todo df_completo):
ViewPosition
AP                147173
PA                 96161
LATERAL            82853
LL                 35133
NaN                15769
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1

ViewPosition de las 55 bad-quality:
ViewPosition
AP         27
LL         22
PA          3
NaN         2
LATERAL     1

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo_view.csv
  shape final: (377110, 25)